#### 29.10.25, &copy; [Evhenii Kostin](https://github.com/DE123MasterProgram2025autumn/DE_Kostin), 2025

# Лабораторна робота №4: Оптимізація коду в R — від циклів до функціонального програмування


__Мета:__ _Освоїти сучасні підходи до оптимізації коду в R, зокрема:_
_- заміну імперативних циклів (for, while, repeat) на функціонали (lapply, sapply, vapply, mapply);_
_- використання векторизованих операцій (rowSums, colMeans, apply);_
_- застосування конструкцій обробки помилок (try, tryCatch) для стійкості конвеєрів._
_Навчитися писати ефективний, читабельний і масштабований код у рамках функціональної парадигми програмування._

### Варіант 2
#### Тема: Аналіз даних про вебсайти з різних джерел

In [2]:
# Виконав(ла): Костін Євгеній | Група: КІ-24-1м | Викладач: Сидоренко В.М.
# Лабораторна робота №4

# --- 0. Підключення бібліотек ---
# Переконайтеся, що ви встановили ці пакети:
install.packages("tidyverse")
install.packages("jsonlite") # Розкоментовано - цей пакет потрібен

library(tidyverse) # Для dplyr, tidyr, purrr, readr
library(jsonlite)  # Для роботи з JSON

# --- ЕТАП 0: Симуляція даних ---
# Цей блок створює тестові дані (10 файлів), як описано у завданні,
# оскільки у нас немає доступу до реальної папки "data/website_data".

message("--- ЕТАП 0: СТВОРЕННЯ ВИХІДНИХ ФАЙЛІВ ---")
data_dir <- "data/website_data"
dir.create(data_dir, recursive = TRUE, showWarnings = FALSE)

# Словник для зберігання даних файлів
files_to_create <- list(
    # 1. Хороший файл
    "site_01.json" = list(site_id = "site_01", month = "2024-01", metrics = list(total_visits = 15200, unique_users = 9800, bounce_rate = 0.42, top_pages = c("home", "contact"))),
    # 2. Хороший файл (той самий сайт, інший місяць)
    "site_02.json" = list(site_id = "site_01", month = "2024-02", metrics = list(total_visits = 16000, unique_users = 10100, bounce_rate = 0.40, top_pages = c("home", "products"))),
    # 3. Хороший файл
    "site_03.json" = list(site_id = "site_02", month = "2024-01", metrics = list(total_visits = 5000, unique_users = 3000, bounce_rate = 0.65, top_pages = c("about"))),
    # 4. Хороший файл
    "site_04.json" = list(site_id = "site_02", month = "2024-02", metrics = list(total_visits = 5500, unique_users = 3200, bounce_rate = 0.60, top_pages = c("about", "blog"))),
    # 5. Хороший файл
    "site_05.json" = list(site_id = "site_03", month = "2024-01", metrics = list(total_visits = 12000, unique_users = 8000, bounce_rate = 0.30, top_pages = c("home", "pricing"))),
    # 8. Хороший файл
    "site_08.json" = list(site_id = "site_01", month = "2024-03", metrics = list(total_visits = 17000, unique_users = 11000, bounce_rate = 0.38, top_pages = c("home"))),
    # 9. Хороший файл
    "site_09.json" = list(site_id = "site_02", month = "2024-03", metrics = list(total_visits = 6000, unique_users = 3500, bounce_rate = 0.58, top_pages = c("blog"))),
    # 7. НЕПОВНИЙ ФАЙЛ (відсутній 'bounce_rate')
    "site_07.json" = list(site_id = "site_05", month = "2024-01", metrics = list(total_visits = 7000, unique_users = 5000)),
    # 10. НЕПОВНИЙ ФАЙЛ (відсутній вузол 'metrics')
    "site_10.json" = list(site_id = "site_06", month = "2024-01")
)

# Запис хороших та неповних JSON-файлів
purrr::iwalk(files_to_create, ~{
    path <- file.path(data_dir, .y)
    jsonlite::write_json(.x, path, auto_unbox = TRUE)
})
message(paste("Створено", length(files_to_create), "JSON файлів."))

# 6. ПОШКОДЖЕНИЙ ФАЙЛ (невалідний JSON)
path <- file.path(data_dir, "site_06.json")
writeLines("{'site_id': 'site_04', 'month': '2024-01', ... ", path)
message("Створено 1 пошкоджений JSON файл.")

message("--- ЕТАП 0 ЗАВЕРШЕНО. ---\n")


# --- Етап 1: Безпечне завантаження та розгортання JSON ---

#' Безпечно читає та розгортає JSON-файл зі статистикою сайту
#'
#' @param path Шлях до JSON-файлу.
#' @return tibble з 5 стовпцями або NULL у разі помилки.
safe_parse_site <- function(path) {
  tryCatch({
    # 1. Прочитати JSON
    data <- jsonlite::read_json(path)
    
    # 2. Використовуємо hoist()
    hoisted_data <- tibble(raw_data = list(data)) %>%
      tidyr::hoist(
        raw_data,
        site_id = "site_id",
        month = "month",
        total_visits = c("metrics", "total_visits"),
        unique_users = c("metrics", "unique_users"),
        bounce_rate = c("metrics", "bounce_rate")
        # .default = NA  <-- ВИДАЛЕНО, оскільки ваша версія tidyr 1.3.1 його не підтримує
      )
    
    # 3. Перевірка наявності всіх полів
    required_fields <- c("site_id", "month", "total_visits", "unique_users", "bounce_rate")
    
    # 3а. НОВИЙ КРОК: Перевіряємо, чи hoist взагалі створив ці стовпці
    #      (Це замінює логіку .default = NA)
    if (!all(required_fields %in% names(hoisted_data))) {
      message(paste("⚠️ Пропущено (відсутні стовпці, напр. 'metrics' або 'bounce_rate'):", path))
      return(NULL)
    }

    # 3b. СТАРИЙ КРОК: Тепер перевіряємо, чи в них є NA (наприклад, якщо б у JSON було "bounce_rate": null)
    if (any(is.na(hoisted_data[required_fields]))) {
      message(paste("⚠️ Пропущено (неповні дані, містить NA):", path))
      return(NULL)
    }
    
    # 4. Повертаємо очищений tibble
    return(hoisted_data)
    
  }, error = function(e) {
    # 5. Обробка помилок (напр., пошкоджений JSON)
    message(paste("❗ Помилка читання файлу:", path, "-", e$message))
    return(NULL)
  })
}


# Отримуємо список усіх JSON-файлів
file_paths <- list.files("data/website_data", 
                         pattern = "\\.json$", 
                         full.names = TRUE)

# Застосовуємо нашу безпечну функцію до кожного файлу за допомогою lapply
list_of_data <- lapply(file_paths, safe_parse_site)

# Фільтруємо NULL-результати (purrr::compact) та об'єднуємо (dplyr::bind_rows)
sites_data <- purrr::compact(list_of_data) %>%
  dplyr::bind_rows() %>%
  # Конвертуємо типи, оскільки hoist міг повернути списки
  mutate(across(c(total_visits, unique_users, bounce_rate), as.numeric))

print("--- Етап 1: Результат завантаження даних (sites_data) ---")
print(sites_data)


# --- Етап 2: Оптимізована обробка даних ---

# 1. Розрахунок 'engagement_rate' (векторизована операція)
sites_data <- sites_data %>%
  mutate(engagement_rate = 1 - bounce_rate)

# 2. Агрегація по 'site_id' (group_by + summarise)
site_summary <- sites_data %>%
  group_by(site_id) %>%
  summarise(
    avg_users = mean(unique_users, na.rm = TRUE),
    max_engagement_rate = max(engagement_rate, na.rm = TRUE),
    months_with_data = n(), # n() рахує кількість рядків у групі
    .groups = 'drop' # Завершуємо групування
  )

print("--- Етап 2: Зведення по сайтах (site_summary) ---")
print(site_summary)


# --- Етап 3: Аналіз якості даних ---

message("\n--- Етап 3: Аналіз якості даних ---")

# 1. Визначте, скільки сайтів мають дані принаймні за 2 місяці
sites_gt_2_months <- site_summary %>%
  filter(months_with_data >= 2)

message(paste("Кількість сайтів з даними >= 2 місяці:", nrow(sites_gt_2_months)))
print(sites_gt_2_months)

# 2. Обчисліть середній bounce rate по всіх сайтах за кожен місяць
monthly_bounce_rate <- sites_data %>%
  group_by(month) %>%
  # Використання summarise(across(...)) за вимогою
  summarise(
    across(bounce_rate, ~mean(.x, na.rm = TRUE))
  ) %>%
  rename(avg_bounce_rate = bounce_rate) # Перейменуємо для ясності

print("\nСередній 'bounce rate' по місяцях:")
print(monthly_bounce_rate)


# --- Етап 4: Порівняння підходів ---

message("\n--- Етап 4: Порівняння підходів ---")
# Завдання: обчислити середній bounce rate для кожного сайту.

# Створимо список датафреймів, по одному для кожного сайту
sites_list <- sites_data %>%
  # group_split створює список з tibbles для кожної групи
  group_split(site_id) %>%
  # Назвемо елементи списку для чистоти результату sapply/vapply
  set_names(sapply(., function(df) df$site_id[1]))

print("Створено список сайтів для ітерації:")
print(names(sites_list))


# --- 4a. Розрахунок за допомогою `for` циклу ---
message("\n--- 4a. Розрахунок за допомогою `for` циклу ---")

# Функція, що використовує цикл
avg_bounce_loop <- function(data_list) {
  # 1. Попереднє виділення пам'яті (важливо для R)
  results <- numeric(length(data_list))
  names(results) <- names(data_list)
  
  # 2. Ітерація з циклом for
  for (i in seq_along(data_list)) {
    site_name <- names(data_list)[i]
    site_data <- data_list[[site_name]]
    results[site_name] <- mean(site_data$bounce_rate, na.rm = TRUE)
  }
  return(results)
}

# Вимірюємо час
time_loop <- system.time(
  loop_results <- avg_bounce_loop(sites_list)
)
print("Результати `for` циклу:")
print(loop_results)
print(time_loop)


# --- 4b. Розрахунок за допомогою `vapply` (функціональний стиль) ---
message("\n--- 4b. Розрахунок за допомогою `vapply` (функціональний стиль) ---")

# `vapply` - найбезпечніша версія, вимагає вказати тип результату (FUN.VALUE)
time_vapply <- system.time(
  vapply_results <- vapply(sites_list, function(site_data) {
    mean(site_data$bounce_rate, na.rm = TRUE)
  }, FUN.VALUE = numeric(1)) # Вказуємо, що чекаємо 1 число
)

print("Результати `vapply`:")
print(vapply_results)
print(time_vapply)

message("\n--- Коментар до Етапу 4 ---")
message("1. Читабельність: Код з `vapply` значно коротший (3 рядки проти 10 для циклу).")
message("2. Безпека: `vapply` є безпечнішим. Ми чітко вказали `FUN.VALUE = numeric(1)`, що гарантує помилку, якщо функція раптом поверне не число.")
message("3. Швидкість: На малих даних (3 сайти) різниця непомітна, але на великих списках `vapply` буде значно швидшим, оскільки сама ітерація відбувається на оптимізованому C-рівні.")

# --- Очищення (необов'язково) ---
# unlink("data", recursive = TRUE)
# message("\nПапку 'data' видалено.")

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

--- ЕТАП 0: СТВОРЕННЯ ВИХІДНИХ ФАЙЛІВ ---

Створено 9 JSON файлів.

Створено 1 пошкоджений JSON файл.

--- ЕТАП 0 ЗАВЕРШЕНО. ---


❗ Помилка читання файлу: data/website_data/site_06.json - lexical error: invalid char in json text.
                                      {'site_id': 'site_04', 'month': 
                     (right here) ------^


⚠️ Пропущено (неповні дані, містить NA): data/website_data/site_07.json

⚠️ Пропущено (неповні дані, містить NA): data/website_data/site_10.json

[1] "--- Етап 1: Результат завантаження даних (sites_data) ---"
# A tibble: 7 × 6
  site_id month   total_visits unique_users bounce_rate raw_data        
  <chr>   <chr>          <dbl>        <dbl>       <dbl> <list>          
1 site_01 2024-01        15200         9800        0.42 <named list [1]>
2 site_01 2024-02        16000        101

<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=6a083947-94ba-475d-8730-27cff0574f54' target="_blank">
 </img>
Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>